# 🎓 DATA PREPARATION GUIDE
## How to Create Training Data for Fine-Tuning

**Critical Note:** The quality of your training data directly determines the quality of your fine-tuned model.  
**Rule of Thumb:** Good data < quantity of data

---

## 📊 Quick Comparison: What Data Format For What?

| Fine-tuning Approach | Data Format | Sample Size | Creation Time |
|:---|:---|:---|:---|
| **Option 1: Embedding** | (question, positive_doc, negative_doc) | 500-2000 | 2-4 weeks |
| **Option 2: QLoRA** | (question, answer) | 200-1000 | 1-3 weeks |
| **Option 4: RAFT** | (q, [docs], correct_answer, [wrong_docs]) | 200-500 | 2-4 weeks |

---

## 🎯 Approach 1: Manual Data Creation (Highest Quality)

### Best For:
- Small datasets (< 500 examples)
- High-quality requirements
- When you have 1-2 weeks

### How To:

In [ ]:
import pandas as pd
from pathlib import Path

# Step 1: Create template for manual data entry
template = pd.DataFrame({
    'question': [
        'Bao nhiêu ngày phép mỗi năm?',
        'Chế độ bảo hiểm là gì?',
        'Lương khởi điểm bao nhiêu?',
    ],
    'answer': [
        'Nhân viên toàn thời gian được 20 ngày phép trả lương mỗi năm theo chính sách công ty.',
        'Công ty cung cấp bảo hiểm y tế toàn diện cho toàn bộ nhân viên và gia đình họ.',
        'Lương khởi điểm phụ thuộc vào trình độ và kinh nghiệm, từ 15-25 triệu đồng/tháng.',
    ]
})

# Save template
template.to_csv('./data/training_template.csv', index=False)
print("✅ Template created: ./data/training_template.csv")
print("\nHow to use:")
print("1. Open in Excel/Google Sheets")
print("2. Add your HR policy questions in 'question' column")
print("3. Add model answers from handbook in 'answer' column")
print("4. Save as training_data.csv")
print("5. Aim for 200-1000 examples")

### Quality Checklist for Manual Data:

In [ ]:
print("""
✅ For EACH question-answer pair, check:

Question Quality:
  □ Is it a realistic question employees might ask?
  □ Is it written in natural Vietnamese (not formal)?
  □ Is it asking ONE thing (not multiple things)?
  □ Would different people ask it in different ways? (add variations)

Answer Quality:
  □ Is the answer directly from the handbook?
  □ Is it complete (not missing key info)?
  □ Is it clear and professional?
  □ Does it match your company's tone?
  □ Can it stand alone (doesn't need extra context)?

Example GOOD pair:
  Q: "Mình muốn xin phép mấy ngày thì được?"
  A: "Theo chính sách công ty, nhân viên toàn thời gian được 20 ngày phép trả lương mỗi năm. 
      Bạn cần báo cáo cho quản lý và submit request qua hệ thống HR."

Example BAD pair:
  Q: "Phép?"
  A: "Xem tài liệu"
  (Too vague, not from handbook directly)
""")

---

## 🤖 Approach 2: LLM-Generated Synthetic Data (Fast & Scalable)

### Best For:
- Creating 500+ examples quickly
- When you need volume
- To augment manual data

### How To:

In [ ]:
# Generate synthetic training data using an LLM
# This uses Google's Generative AI (free tier available)

import google.generativeai as genai

# Configure (you need a Google API key)
# Get it free from: https://makersuite.google.com/app/apikey
# genai.configure(api_key="YOUR_API_KEY")

def generate_qa_pairs(handbook_text, num_pairs=50):
    """Generate Q&A pairs from handbook text"""
    
    prompt = f"""Given the following HR handbook content:

{handbook_text}

Generate {num_pairs} realistic Q&A pairs that employees might ask about.
Format: question\t|\tanswer
Requirements:
- Questions should be natural Vietnamese (casual, realistic)
- Answers should directly come from the handbook
- Each pair on separate line
- Questions should vary (different phrasings)
- Answers should be complete but concise
"""
    
    try:
        model = genai.GenerativeModel('gemini-1.5-flash')
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"Error: {e}")
        print("Make sure to set your API key")
        return None

# Example handbook snippet
handbook = """CHÍNH SÁCH NGHỈ PHÉP
- Nhân viên toàn thời gian: 20 ngày/năm
- Nhân viên bán thời gian: 10 ngày/năm
- Phải báo cáo trước 3 ngày
- Phải được sếp phê duyệt
"""

print("To use this approach:")
print("""
1. Get Google API key (free): https://makersuite.google.com/app/apikey
2. Set your key: genai.configure(api_key="YOUR_KEY")
3. Extract handbook sections to text
4. Call generate_qa_pairs() for each section
5. Save results to CSV
""")

---

## 📚 Approach 3: Hybrid - Manual + LLM-Generated

### Best For:
- Balancing quality and speed
- Creating 1000+ high-quality examples
- Recommended approach!

### How To:

In [ ]:
# Step 1: Create 100-200 manual pairs (high quality baseline)
# Step 2: Use LLM to generate 3-4 variations of each (synthetic augmentation)
# Step 3: Review + filter LLM-generated pairs
# Step 4: Combine into final training set

import pandas as pd

# Load manually created data
manual_data = pd.read_csv('./data/manual_qa_pairs.csv')
print(f"Manual pairs: {len(manual_data)}")

# Simulate LLM-generated variations
def generate_question_variations(original_question, num_variations=3):
    """Generate alternative phrasings of the same question"""
    # In real implementation, use LLM here
    variations = [
        original_question,  # Keep original
        f"Cho hỏi về: {original_question.lower()}",
        f"Mình muốn biết {original_question.lower()}?",
    ]
    return variations[:num_variations]

# Create augmented dataset
augmented_data = []
for idx, row in manual_data.iterrows():
    q = row['question']
    a = row['answer']
    
    # Add original
    augmented_data.append({'question': q, 'answer': a})
    
    # Add variations
    for var_q in generate_question_variations(q, 2):
        if var_q != q:
            augmented_data.append({'question': var_q, 'answer': a})

augmented_df = pd.DataFrame(augmented_data)
print(f"After augmentation: {len(augmented_df)} pairs")
print(f"Amplification factor: {len(augmented_df) / len(manual_data):.1f}x")

---

## 🔍 Step 3: Data Quality Validation

In [ ]:
import pandas as pd

def validate_training_data(df):
    """Check quality of training data"""
    issues = []
    
    for idx, row in df.iterrows():
        q = str(row.get('question', ''))
        a = str(row.get('answer', ''))
        
        # Check question
        if len(q) < 5:
            issues.append(f"Row {idx}: Question too short")
        if len(q) > 500:
            issues.append(f"Row {idx}: Question too long")
        if not q.endswith(('?', '。', '?')):
            issues.append(f"Row {idx}: Question doesn't end with '?'")
        
        # Check answer
        if len(a) < 20:
            issues.append(f"Row {idx}: Answer too short")
        if len(a) > 2000:
            issues.append(f"Row {idx}: Answer too long")
        if a.lower().startswith(('unknown', 'i don\'t', 'not sure')):
            issues.append(f"Row {idx}: Answer seems uncertain")
    
    return issues

# Validate your data
# issues = validate_training_data(your_dataframe)
# if issues:
#     print(f"⚠️  Found {len(issues)} issues:")
#     for issue in issues[:10]:
#         print(f"  - {issue}")
# else:
#     print("✅ Data looks good!")

---

## 📋 Step 4: Train/Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

def prepare_train_val_split(df, test_size=0.1):
    """Split data into training and validation sets"""
    
    train_df, val_df = train_test_split(
        df,
        test_size=test_size,
        random_state=42
    )
    
    # Save both
    train_df.to_csv('./data/training_data_train.csv', index=False)
    val_df.to_csv('./data/training_data_val.csv', index=False)
    
    print(f"✅ Split complete:")
    print(f"  Training: {len(train_df)} pairs (90%)")
    print(f"  Validation: {len(val_df)} pairs (10%)")
    
    return train_df, val_df

---

## 📊 Summary: Data Sizes and Expected Results

| Pairs | Quality | Effort | Result | Time |
|:---|:---|:---|:---|:---|
| **50-100** | Must be high | Low | Marginal improvement | 3-5 days |
| **100-300** | High | Medium | Good improvement | 1-2 weeks |
| **300-1000** | Good | High | Great improvement | 2-4 weeks |
| **1000+** | OK | Very High | Excellent improvement | 4+ weeks |

---

## 💡 Tips for Best Results

1. **Quality over Quantity**
   - 200 perfect pairs > 1000 mediocre pairs
   - Every pair should be high quality

2. **Variety Matters**
   - Include different question styles
   - Include different answer lengths
   - Cover all major policies

3. **Real World Phrasing**
   - Use casual Vietnamese (not formal)
   - Include variations and typos
   - Think like an employee

4. **Verify Against Handbook**
   - Every answer should be traceable to handbook
   - If handbook doesn't cover it, don't include

5. **Document Your Process**
   - Note sources (which handbook section)
   - Note who created each pair
   - This helps with future refinement

---

## 🎯 Recommended: Start Here

1. **Week 1:** Manually create 200-300 high-quality pairs
2. **Week 2:** Use LLM to generate 2-3 variations of each
3. **Week 3:** Review, validate, and filter
4. **Week 4:** Fine-tune your model

**Result:** 600-900 training pairs of high quality